### Query Enhancement - HyDE

What is HyDE? <br>
HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user's query directly, you first generate a hypothetical answer (document) to the query using an LLM - and then embed that hypothetical document to search your vector store.

Hypothetical bridges the gap between user intent and relevant content, especially when:
1. Queries are short.
2. Language mismatch between query and documents.
3. You want to retrieve based on answer content, not question words.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [2]:
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

In [3]:
# Load documents from Wikipedia
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

# splitting data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)

docs


[Document(page_content="Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the company's", metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation

In [4]:
# Step 2: Create Vector Store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embedding_model)
vector_store

c:\RAG\env_langchain_lessthan1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\RAG\env_langchain_lessthan1\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
# Step 4: Create LLM to generate hypothetical answers
llm = ChatOpenAI(
    model_name="o4-mini",
    temperature=1
)

c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [6]:
from langchain.vectorstores import Chroma

# Creating vector store with Chroma
db = Chroma.from_documents(docs, embedding_model, persist_directory="output/steve_jobs_for_hyde.db")

# Create the retriever
base_retriever = db.as_retriever(search_kwargs={"k": 5})
base_retriever

Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001F6E88F3D90>, search_kwargs={'k': 5})

In [7]:
# Generating a prompt for generating hypothetical answers
from langchain.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate


def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'. 
    Create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_messages(query=query)
    print(messages)

    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc


In [8]:
query = "When was Steve Jobs fired from Apple?"
hyde_doc = get_hyde_doc(query)
print(hyde_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'. \n    Create a hypothetical answer for the topic")]
Steve Jobs’s departure from Apple was not a single dramatic moment but the culmination of mounting tensions with the board and then-CEO John Sculley during the spring of 1985.  Here’s a concise yet detailed timeline and explanation:

1. Early Growth and Rising Friction (1983–1984)  
   • 1983: Apple co-founder Steve Jobs (age 28) recruits Pepsi executive John Sculley to be Apple’s CEO, hoping Sculley’s marketing prowess will accelerate the young company’s expansion.  
   • Late 1983 into 1984: The two clash over product strategy, resource allocation and corporate structure. Jobs champions an updated, lower-cost Macintosh to extend Apple’s reach; Sculley wants to protect margins on the Lisa and premium-priced Mac.

2. Macintosh Launch and Aftermath (January 1984 – Early 1985)  
   • January 24, 1984: T

### This answer will forwarded to the Retriever, but before this we have to go and apply embeddings on this.

In [9]:
matched_doc = base_retriever.invoke(get_hyde_doc(query))
print(matched_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'. \n    Create a hypothetical answer for the topic")]
[Document(page_content="In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer platform development company that specialized in computers for higher-education and business markets,", metadata={'source': 'https://en.wikipedia.org/wiki/Steve_Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its ch

After retriever, we can do after that.